<a href="https://colab.research.google.com/github/Yashwant-Vadhan/Indian-Cattle-Breeds_SIH25004/blob/Image-Cropping-%26-Breed-filtering/Data_processing_(Crop_%26_Filter).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Target Indian cattle breeds
cattle_breeds = [
    "Alambadi cattle",
    "Amrit Mahal cattle",
    "Bachaur cattle",
    "Bargur cattle",
    "Dangi cattle",
    "Deoni cattle",
    "Gaolao cattle",
    "Gidda cattle",
    "Gir cattle",
    "Hallikar cattle",
    "Hariana cattle",
    "Kangayam cattle",
    "Kankrej cattle",
    "Kasaragod Dwarf",
    "Kenkatha cattle",
    "Kherigarh cattle",
    "Khillari cattle",
    "Krishna Valley cattle",
    "Malvi cattle",
    "Mewati cattle",
    "Nagori cattle",
    "Nimari cattle",
    "Ongole cattle",
    "Ponwar cattle",
    "Pulikulam cattle",
    "Rathi cattle",
    "Red Kandhari cattle",
    "Red Sindhi cattle",
    "Sahiwal cattle",
    "Siri cattle",
    "Tharparkar cattle",
    "Vechur cattle",
    "Motu cattle",
    "Ghumusari cattle",
    "Binjharpuri cattle",
    "Khariar cattle",
    "Kosali cattle",
    "Belahi cattle",
    "Gangatiri cattle",
    "Badri cattle",
    "Lakhimi cattle",
    "Ladakhi cattle",
    "Konkan Kapila cattle",
    "Poda Thurpu cattle",
    "Nari cattle",
    "Dagri cattle",
    "Thutho cattle",
    "Shweta Kapila cattle",
    "Himachali Pahari cattle",
    "Purnea cattle",
    "Umblachery cattle"
]

In [3]:
# 1) Install Dependencies
!pip install -q ultralytics pillow tqdm transformers torch torchvision --upgrade

# 2) Imports & Setup
import os
import shutil
from glob import glob
from PIL import Image
from tqdm import tqdm

import torch
from ultralytics import YOLO

from transformers import CLIPProcessor, CLIPModel
from PIL import Image as PILImage

# Prefer GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_PATH = "/content/drive/MyDrive/Cattle_Dataset_copy"
# Where to put cropped images (mirrors the same subfolder layout)
CROPPED_BASE = "/content/drive/MyDrive/Cattle_Dataset_cropped"  # <-- CHANGE IF YOU LIKE
os.makedirs(CROPPED_BASE, exist_ok=True)

# Cropping options
KEEP_ONLY_LARGEST_COW = True   # if True: keep only the largest cow per image; else: save all cow boxes
RESIZE_CROPS = True            # if True: resize crops to (224, 224)
RESIZE_SIZE = (224, 224)

# CLIP filtering options
CLIP_THRESHOLD = 0.35  # your existing threshold
USE_GENERIC_FALLOFF = 0.5

# 4) Load Models (YOLO + CLIP)
# YOLOv8 (COCO-pretrained: includes 'cow')
yolo_model = YOLO("yolov8s.pt")  # 'n' is faster, 's' is a bit better; change if you need

# CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# 5) Cow Cropping Utilities
def _get_cow_boxes(result):
    """
    Returns list of (xyxy_tensor, cls_id_int, conf_float) for boxes whose class name == 'cow'
    Uses result.names mapping to avoid hardcoding COCO indices.
    """
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return []

    out = []
    names = result.names  # dict: {class_id: "name"}
    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i].item())
        name = names.get(cls_id, "")
        if name == "cow":
            xyxy = boxes.xyxy[i]
            conf = float(boxes.conf[i].item()) if boxes.conf is not None else 0.0
            out.append((xyxy, cls_id, conf))
    return out


def crop_cows_from_image(image_path, output_dir, keep_only_largest=True, resize=False, resize_size=(224, 224)):
    """
    Detect cows in image_path, crop them, and save to output_dir.
    Returns number of crops saved.
    """
    try:
        results = yolo_model(image_path, verbose=False)
        if len(results) == 0:
            return 0
        result = results[0]
        cow_boxes = _get_cow_boxes(result)
        if not cow_boxes:
            return 0

        img = Image.open(image_path).convert("RGB")
        base = os.path.splitext(os.path.basename(image_path))[0]

        # If we only want the biggest cow, choose the box with largest area
        if keep_only_largest:
            def area(xyxy):
                x1, y1, x2, y2 = map(float, xyxy.tolist())
                return max(0.0, x2 - x1) * max(0.0, y2 - y1)
            cow_boxes = [max(cow_boxes, key=lambda t: area(t[0]))]

        saved = 0
        for idx, (xyxy, cls_id, conf) in enumerate(cow_boxes):
            x1, y1, x2, y2 = map(int, xyxy.tolist())
            x1 = max(0, x1); y1 = max(0, y1); x2 = min(img.width, x2); y2 = min(img.height, y2)
            if x2 <= x1 or y2 <= y1:
                continue

            crop = img.crop((x1, y1, x2, y2))
            if resize:
                crop = crop.resize(resize_size, Image.BICUBIC)

            os.makedirs(output_dir, exist_ok=True)
            save_path = os.path.join(output_dir, f"{base}_cow{idx}.jpg")
            crop.save(save_path, quality=95)
            saved += 1

        return saved
    except Exception as e:
        print(f"[crop_cows_from_image] Error for {image_path}: {e}")
        return 0


def batch_crop_cows_for_breed(input_folder, output_folder,
                              keep_only_largest=True,
                              resize=False,
                              resize_size=(224, 224)):
    """
    Walks input_folder and crops cows into output_folder.
    """
    os.makedirs(output_folder, exist_ok=True)
    image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    files = [f for f in os.listdir(input_folder) if f.lower().endswith(image_exts)]

    total_crops = 0
    for f in tqdm(files, desc=f"Cropping cows in {os.path.basename(input_folder)}"):
        in_path = os.path.join(input_folder, f)
        total_crops += crop_cows_from_image(
            in_path, output_folder,
            keep_only_largest=keep_only_largest,
            resize=resize,
            resize_size=resize_size
        )

    print(f"✅ {os.path.basename(input_folder)}: saved {total_crops} cropped cow(s) → {output_folder}")
    return total_crops

# 6) Your CLIP Breed Filter (integrated)
def is_correct_breed(image_path, breed_name, threshold=CLIP_THRESHOLD, generic_falloff=USE_GENERIC_FALLOFF):
    """
    Your original logic, kept intact, with a small robustness tweak.
    """
    try:
        image = PILImage.open(image_path).convert("RGB")

        # Positive prompts for the target breed
        positive_prompts = [
            f"a photo of a {breed_name} cow",
            f"a photo of a {breed_name} bull",
            f"a photo of {breed_name} cattle",
        ]

        # Generic cattle prompts (backup positives)
        generic_prompts = [
            "a photo of a cow",
            "a photo of a bull",
            "a photo of cattle",
        ]

        # Negative prompts to filter junk
        negative_prompts = [
            "photo of captions", "photo of quotes","photo of sentence",
            "photo of persons with cow",
            "a photo of a elephant", "photo of ghee bottle",
            "photo of dairy products",
            "a photo of a person",
            "a photo of humans",
            "a photo of food",
            "a photo of a box",
            "a photo of a building",
            "a photo of scenery",
            "a photo of a goat",
            "a photo of a buffalo",
            "a photo of a horse",
            "not a cow",
        ]

        all_prompts = positive_prompts + generic_prompts + negative_prompts

        inputs = clip_processor(
            text=all_prompts,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(DEVICE)

        outputs = clip_model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1).detach().cpu().numpy()[0]

        # Slices
        p_len = len(positive_prompts)
        g_len = len(generic_prompts)

        breed_score = float(probs[:p_len].sum())  # breed-specific
        cattle_score = float(probs[p_len:p_len+g_len].sum())  # generic cattle

        final_score = breed_score + generic_falloff * cattle_score
        return final_score > threshold

    except Exception as e:
        # print(f"[is_correct_breed] Error for {image_path}: {e}")
        return False


def filter_wrong_breeds(folder_path, breed_name):
    removed = 0
    kept = 0
    image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(image_exts)]

    for file in tqdm(files, desc=f"CLIP filter: {os.path.basename(folder_path)}"):
        file_path = os.path.join(folder_path, file)
        if not is_correct_breed(file_path, breed_name):
            try:
                os.remove(file_path)
                removed += 1
            except Exception as e:
                print(f"[filter_wrong_breeds] Could not remove {file_path}: {e}")
        else:
            kept += 1

    print(f"🧹 {os.path.basename(folder_path)}: kept {kept}, removed {removed} (after CLIP filter)")
    return kept, removed

# 7) PIPELINE
#    (A) Crop cows → (B) CLIP filter crops
def run_pipeline_for_breed(breed_name):
    """
    1) Reads from BASE_PATH/<breed_name> (raw images)
    2) Writes cropped cows to CROPPED_BASE/<breed_name>
    3) CLIP-filters the cropped cows in-place
    """
    in_dir = os.path.join(BASE_PATH, breed_name.replace(" ", "_"))
    out_dir = os.path.join(CROPPED_BASE, breed_name.replace(" ", "_"))

    if not os.path.isdir(in_dir):
        print(f"⚠ Input folder missing: {in_dir}")
        return

    # A) Crop cows
    batch_crop_cows_for_breed(
        in_dir,
        out_dir,
        keep_only_largest=KEEP_ONLY_LARGEST_COW,
        resize=RESIZE_CROPS,
        resize_size=RESIZE_SIZE
    )

    # B) CLIP breed filter on the cropped set
    filter_wrong_breeds(out_dir, breed_name)


def run_pipeline_all_breeds():
    for breed in cattle_breeds:
        print("\n" + "="*80)
        print(f"▶ Processing breed: {breed}")
        run_pipeline_for_breed(breed)
    print("\n🎉 Done!")

# 8) RUNNING THE FUNCTION TO FILTER AND CROP
run_pipeline_all_breeds()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 68.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

🧹 Purnea_cattle: kept 121, removed 2 (after CLIP filter)

▶ Processing breed: Umblachery cattle


Cropping cows in Umblachery_cattle: 100%|██████████| 274/274 [04:29<00:00,  1.02it/s]


✅ Umblachery_cattle: saved 243 cropped cow(s) → /content/drive/MyDrive/Cattle_Dataset_cropped/Umblachery_cattle


CLIP filter: Umblachery_cattle: 100%|██████████| 231/231 [02:50<00:00,  1.36it/s]

🧹 Umblachery_cattle: kept 218, removed 13 (after CLIP filter)

🎉 Done!
